# **Random Seed**

In [1]:
import numpy as np
import tensorflow as tf
import random

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# Data Preprocessing and Importing

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from keras.models import Model
from keras.layers import Input, Conv1D, LSTM, Dense, Concatenate
from keras.layers import GlobalAveragePooling1D, Multiply, Dropout
from keras.layers import BatchNormalization, Softmax
from keras.optimizers import Adam

In [3]:
df = pd.read_csv("wustl-ehms-2020_with_attacks_categories.csv")

drop_cols = ['Dir', 'SrcAddr', 'DstAddr', 'SrcMac', 'DstMac', 'Sport', 'Dport', 'Label', 'Packet_num']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

if 'Flgs' in df.columns:
    le_flags = LabelEncoder()
    df['Flgs'] = le_flags.fit_transform(df['Flgs'].astype(str))

target_enc = LabelEncoder()
y = target_enc.fit_transform(df['Attack Category'])

X_df = df.drop(columns=['Attack Category'], errors='ignore')
X_df = X_df.select_dtypes(include=[np.number]).fillna(0)
X = X_df.values

In [4]:
indices = np.arange(len(df))

idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices, y, test_size=0.3, stratify=y, random_state=42
)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

scaler = StandardScaler()

scaler.fit(X[idx_train])

X_scaled = scaler.transform(X)

weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(weights))



In [5]:
normal_id = list(target_enc.classes_).index('normal')
attack_ids_in_test = np.where(y_test != normal_id)[0]
random_test_idx = np.random.choice(attack_ids_in_test)
global_idx = idx_test[random_test_idx]

test_sample_1d = X_scaled[global_idx]
true_label = y_test[random_test_idx]

test_sample_3d = test_sample_1d.reshape(1, X_scaled.shape[1], 1)


In [6]:
print("--- Preprocessing Complete ---")
print(f"Total dataset shape: {X_scaled.shape}")
print(f"Classes: {target_enc.classes_}")
print(f"Train/Val/Test samples: {len(idx_train)} / {len(idx_val)} / {len(idx_test)}")
print(f"\nRandom Attack Sample Selected:")
print(f"True Label ID: {true_label} ({target_enc.inverse_transform([true_label])[0]})")
print(f"3D Sample Shape: {test_sample_3d.shape}")

--- Preprocessing Complete ---
Total dataset shape: (16318, 35)
Classes: ['Data Alteration' 'Spoofing' 'normal']
Train/Val/Test samples: 11422 / 2448 / 2448

Random Attack Sample Selected:
True Label ID: 0 (Data Alteration)
3D Sample Shape: (1, 35, 1)


# **CNN + LSTM**

In [7]:
inputs = Input(shape=(X_scaled.shape[1], 1))

cnn_b = Conv1D(128, 3, activation='relu', padding='same')(inputs)
cnn_a = Softmax(axis=1)(Dense(1, activation='tanh')(cnn_b))
cnn = Dense(42, activation='relu')(GlobalAveragePooling1D()(Multiply()([cnn_b, cnn_a])))

In [8]:
lstm_b = LSTM(64, return_sequences=True)(inputs)
lstm_a = Softmax(axis=1)(Dense(1, activation='tanh')(lstm_b))
lstm = Dense(42, activation='relu')(GlobalAveragePooling1D()(Multiply()([lstm_b, lstm_a])))


In [9]:
fused = Multiply()([cnn, lstm])
merged = Concatenate()([cnn, lstm, fused])
merged = Dropout(0.1105123161523788)(BatchNormalization()(merged))

In [10]:
from keras.callbacks import EarlyStopping

out = Dense(len(target_enc.classes_), activation='softmax')(Dense(42, activation='relu')(merged))
keras_model = Model(inputs, out)
keras_model.compile(optimizer=Adam(learning_rate=0.008268126053057996), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

print("\nTraining CNN-LSTM Model...")

X_train_cnn = X_scaled[idx_train].reshape(-1, X_scaled.shape[1], 1)

history = keras_model.fit(X_train_cnn, y_train, epochs=100, batch_size=64, validation_split=0.2, class_weight=class_weights_dict,
                    callbacks=[early_stop],
                    verbose=1)


Training CNN-LSTM Model...
Epoch 1/100
143/143 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.5529 - loss: 0.5046 - val_accuracy: 0.8731 - val_loss: 0.6522
Epoch 2/100
143/143 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step - accuracy: 0.6794 - loss: 0.4285 - val_accuracy: 0.9326 - val_loss: 0.7684
Epoch 3/100
143/143 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - accuracy: 0.6797 - loss: 0.4223 - val_accuracy: 0.2810 - val_loss: 0.8185
Epoch 4/100
143/143 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.6694 - loss: 0.4099 - val_accuracy: 0.6814 - val_loss: 0.6707
Epoch 5/100
143/143 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.6783 - loss: 0.4046 - val_accuracy: 0.4551 - val_loss: 1.0072
Epoch 6/100
143/143 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step - accuracy: 0.6691 - loss: 0.4091 - val_accuracy: 0.4162 - val_loss: 0.8922
Epoch 7/100
143/143 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.6728 - loss: 0.4050 - val_accuracy: 0.9348 - val_loss: 0.2385
Epoch 8/100
143/143 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accur

In [11]:
prediction_probs = keras_model.predict(test_sample_3d, verbose=0)[0]
predicted_label = np.argmax(prediction_probs)

print("\nEvaluating Model on Test Data...")

X_test_cnn = X_scaled[idx_test].reshape(-1, X_scaled.shape[1], 1)

loss, accuracy = keras_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f" FINAL TEST ACCURACY: {accuracy * 100:.2f}% ")

print(f"\nPredicted : {target_enc.classes_[predicted_label].upper()}")
print(f"Truth     : {target_enc.classes_[true_label].upper()}")


Evaluating Model on Test Data...
 FINAL TEST ACCURACY: 93.67% 

Predicted : DATA ALTERATION
Truth     : DATA ALTERATION


# **GNN**

In [13]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.2 MB/s eta 0:00:00


In [14]:
import torch
import numpy as np
from torch_geometric.data import Data
from sklearn.neighbors import NearestNeighbors

# 1. Create Boolean Masks from our unified indices
num_nodes = len(X_scaled)
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[idx_train] = True
val_mask[idx_val] = True
test_mask[idx_test] = True

# 2. Build Temporal Edges (Connecting i to i+1)
source_nodes = np.arange(num_nodes - 1)
target_nodes = np.arange(1, num_nodes)
temporal_edge_index = np.vstack((source_nodes, target_nodes))
temporal_edge_index = np.concatenate([temporal_edge_index, temporal_edge_index[::-1]], axis=1)

# 3. Build KNN Edges
print("Computing KNN edges...")
k = 5
knn = NearestNeighbors(n_neighbors=k+1, metric='cosine', n_jobs=-1)
knn.fit(X_scaled)
distances, indices = knn.kneighbors(X_scaled)

knn_sources = np.repeat(np.arange(num_nodes), k)
knn_targets = indices[:, 1:].flatten()
knn_edge_index = np.vstack((knn_sources, knn_targets))
knn_edge_index = np.concatenate([knn_edge_index, knn_edge_index[::-1]], axis=1)

# 4. Combine edges and create PyG Data object
combined_edges = np.concatenate([temporal_edge_index, knn_edge_index], axis=1)
edge_index_unique = np.unique(combined_edges, axis=1)

data = Data(
    x=torch.tensor(X_scaled, dtype=torch.float),
    edge_index=torch.tensor(edge_index_unique, dtype=torch.long),
    y=torch.tensor(y, dtype=torch.long)
)
data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print(f"Graph constructed successfully! Nodes: {data.num_nodes}, Edges: {data.num_edges}")

Computing KNN edges...
Graph constructed successfully! Nodes: 16318, Edges: 128208


# **GAT**

In [20]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class GAT_v2(torch.nn.Module):
    def __init__(self, num_features, num_classes):
        super(GAT_v2, self).__init__()

        # Layer 1: heads=4, output = 64 * 4 = 256
        self.conv1 = GATConv(num_features, 64, heads=4, dropout=0.21001645973261923)
        self.bn1 = torch.nn.BatchNorm1d(64 * 4)

        # Layer 2: Input = 256. heads=4, output = 32 * 4 = 128
        self.conv2 = GATConv(64 * 4, 32, heads=4, dropout=0.21001645973261923)
        self.bn2 = torch.nn.BatchNorm1d(32 * 4)

        # Layer 3: Input = 128. heads=1
        self.conv3 = GATConv(32 * 4, num_classes, heads=1, concat=False, dropout=0.21001645973261923)
        self.dropout = torch.nn.Dropout(p=0.21001645973261923)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.conv3(x, edge_index)
        return x

print("GAT_v2 Architecture Defined!")

GAT_v2 Architecture Defined!


In [21]:
import torch.optim as optim
import copy

print("Initializing GAT_v2 Model...")
# Using data.x.shape[1] makes it compatible with both Approach 1 (Raw) and Approach 2 (Embeddings)
gat_model = GAT_v2(num_features=data.x.shape[1], num_classes=len(target_enc.classes_))

# Using your highly optimized learning rate and weight decay
gat_optimizer = optim.Adam(gat_model.parameters(), lr=0.009640661422823864, weight_decay=2.5317495991481588e-05)
class_weights_tensor = torch.tensor(weights, dtype=torch.float)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

best_val_acc = 0
best_model_state = None
patience_counter = 0

print("🔥 Training GAT v2 (with BatchNorm) 🔥")
for epoch in range(1, 2001):
    # --- TRAIN ---
    gat_model.train()
    gat_optimizer.zero_grad()
    out = gat_model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    gat_optimizer.step()

    # --- EVALUATE ---
    gat_model.eval()
    with torch.no_grad():
        out = gat_model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        val_acc = (pred[data.val_mask] == data.y[data.val_mask]).sum().item() / data.val_mask.sum().item()

    # --- EARLY STOPPING ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(gat_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 50 == 0:
        print(f'Epoch: {epoch:03d} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}')

    if patience_counter >= 200:
        print(f'\nEarly stopping at epoch {epoch}! Restoring best weights (Val Acc: {best_val_acc:.4f}).')
        break

# Load the best weights back into the model for testing
if best_model_state is not None:
    gat_model.load_state_dict(best_model_state)

Initializing GAT_v2 Model...
🔥 Training GAT v2 (with BatchNorm) 🔥
Epoch: 050 | Loss: 0.3249 | Val Acc: 0.8178
Epoch: 100 | Loss: 0.3029 | Val Acc: 0.8435
Epoch: 150 | Loss: 0.2827 | Val Acc: 0.8533
Epoch: 200 | Loss: 0.2846 | Val Acc: 0.8452
Epoch: 250 | Loss: 0.2808 | Val Acc: 0.8595
Epoch: 300 | Loss: 0.2565 | Val Acc: 0.8403
Epoch: 350 | Loss: 0.2549 | Val Acc: 0.8546
Epoch: 400 | Loss: 0.2522 | Val Acc: 0.8456
Epoch: 450 | Loss: 0.2399 | Val Acc: 0.7978

Early stopping at epoch 488! Restoring best weights (Val Acc: 0.8930).


In [22]:
from sklearn.metrics import classification_report
import torch.nn.functional as F
import torch

# 1. Final Test Accuracy on Unseen Data
gat_model.eval()
with torch.no_grad():
    out = gat_model(data.x, data.edge_index)
    test_logits = out[data.test_mask]
    test_preds = test_logits.argmax(dim=1).numpy()
    test_true = data.y[data.test_mask].numpy()

test_acc = (test_preds == test_true).sum() / len(test_true)
print(f'\n🏆 GAT_v2 FINAL TEST ACCURACY: {test_acc * 100:.2f}% 🏆\n')

# 2. Classification Report
target_names = [str(c) for c in target_enc.classes_]
print("Classification Report on GAT_v2 Test Set:")
print(classification_report(test_true, test_preds, target_names=target_names))

# ==========================================
# 3. Detailed Check of your Specific Example
# ==========================================
# We find where this specific node lives inside the PyTorch test set
target_local_idx = np.where(idx_test == global_idx)[0][0]

# Extract the raw numbers and the true label for this specific packet
sample_logits = test_logits[target_local_idx]
true_label_id = test_true[target_local_idx]

# Apply softmax to see GAT's exact confidence percentages
sample_probs = F.softmax(sample_logits, dim=0).numpy()
gat_pred = np.argmax(sample_probs)

print("\n" + "="*50)
print(f" GAT_v2 BREAKDOWN FOR TEST SAMPLE {global_idx}")
print("="*50)
print(f"GAT Probabilities     : {np.round(sample_probs, 4)}")
print("-" * 50)
print(f"Final Prediction      : {target_enc.classes_[gat_pred].upper()}")
print(f"Truth                 : {target_enc.classes_[true_label_id].upper()}")
print("="*50)


🏆 GAT_v2 FINAL TEST ACCURACY: 88.77% 🏆

Classification Report on GAT_v2 Test Set:
                 precision    recall  f1-score   support

Data Alteration       1.00      1.00      1.00       139
       Spoofing       0.34      0.66      0.45       168
         normal       0.97      0.90      0.93      2141

       accuracy                           0.89      2448
      macro avg       0.77      0.85      0.79      2448
   weighted avg       0.93      0.89      0.90      2448


 GAT_v2 BREAKDOWN FOR TEST SAMPLE 1012
GAT Probabilities     : [2.000e-04 3.678e-01 6.320e-01]
--------------------------------------------------
Final Prediction      : NORMAL
Truth                 : NORMAL


# **Ensemble Hybrid**

In [23]:
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import classification_report
import torch

print("--- Commencing Late-Fusion Ensemble (Keras CNN+LSTM & PyTorch GAT_v2) ---")

# ==========================================
# 1. Keras CNN+LSTM Probabilities
# ==========================================
# We use the PyTorch test_mask to slice the raw data. This guarantees that
# Keras evaluates the exact same packets in the exact same order as the Graph!
X_test_keras = X_scaled[data.test_mask.numpy()].reshape(-1, X_scaled.shape[1], 1)
keras_probs = keras_model.predict(X_test_keras, verbose=0)

# ==========================================
# 2. PyTorch GAT_v2 Probabilities
# ==========================================
gat_model.eval()
with torch.no_grad():
    out = gat_model(data.x, data.edge_index)
    test_logits = out[data.test_mask]
    gat_probs = F.softmax(test_logits, dim=1).numpy()

# ==========================================
# 3. Average the Probabilities (Late Fusion)
# ==========================================
ensemble_probs = (keras_probs + gat_probs) / 2.0

# Final Predictions
ensemble_preds = np.argmax(ensemble_probs, axis=1)
test_true = data.y[data.test_mask].numpy()

# ==========================================
# 4. Evaluate the Ensemble!
# ==========================================
ensemble_acc = (ensemble_preds == test_true).sum() / len(test_true)
print(f"\n🏆 CNN+LSTM + GAT_v2 ENSEMBLE ACCURACY: {ensemble_acc * 100:.2f}% 🏆\n")

target_names = [str(c) for c in target_enc.classes_]
print("Classification Report on Ensemble:")
print(classification_report(test_true, ensemble_preds, target_names=target_names))

# ==========================================
# 5. Check our Specific Test Sample!
# ==========================================
# We use the EXACT same 'global_idx' packet from your header memory
test_indices = np.where(data.test_mask.numpy())[0]
target_local_idx = np.where(test_indices == global_idx)[0][0]

# Extract the probabilities and the TRUE LABEL for this specific packet
sample_keras_prob = keras_probs[target_local_idx]
sample_gat_prob = gat_probs[target_local_idx]
sample_ensemble_prob = ensemble_probs[target_local_idx]
true_label_id = test_true[target_local_idx]  # Dynamically pull the truth label

ensemble_final_pred = np.argmax(sample_ensemble_prob)

print("\n" + "="*50)
print(f" ENSEMBLE BREAKDOWN FOR TEST SAMPLE {global_idx}")
print("="*50)
print(f"Keras CNN+LSTM Probabilities : {np.round(sample_keras_prob, 4)}")
print(f"PyTorch GAT_v2 Probabilities : {np.round(sample_gat_prob, 4)}")
print(f"Ensemble Average Probabilities: {np.round(sample_ensemble_prob, 4)}")
print("-" * 50)
print(f"Final Prediction      : {target_enc.classes_[ensemble_final_pred].upper()}")
print(f"Truth                 : {target_enc.classes_[true_label_id].upper()}")
print("="*50)

--- Commencing Late-Fusion Ensemble (Keras CNN+LSTM & PyTorch GAT_v2) ---

🏆 CNN+LSTM + GAT_v2 ENSEMBLE ACCURACY: 94.61% 🏆

Classification Report on Ensemble:
                 precision    recall  f1-score   support

Data Alteration       0.97      1.00      0.99       139
       Spoofing       0.75      0.36      0.48       168
         normal       0.95      0.99      0.97      2141

       accuracy                           0.95      2448
      macro avg       0.89      0.78      0.81      2448
   weighted avg       0.94      0.95      0.94      2448


 ENSEMBLE BREAKDOWN FOR TEST SAMPLE 1012
Keras CNN+LSTM Probabilities : [0.0009 0.2213 0.7777]
PyTorch GAT_v2 Probabilities : [1.000e-04 9.435e-01 5.650e-02]
Ensemble Average Probabilities: [5.000e-04 5.824e-01 4.171e-01]
--------------------------------------------------
Final Prediction      : SPOOFING
Truth                 : SPOOFING


# **Feature Pipeline Method**

In [24]:
import torch
import numpy as np
from tensorflow.keras.models import Model

# 1. EXTRACT KERAS EMBEDDINGS
embedding_model = Model(inputs=keras_model.input, outputs=keras_model.layers[-2].output)
X_3d_keras = X_scaled.reshape(-1, X_scaled.shape[1], 1)
cnn_lstm_embeddings = embedding_model.predict(X_3d_keras, verbose=0)

# 2. COMBINE FEATURES & UPDATE GRAPH
combined_features = np.concatenate([X_scaled, cnn_lstm_embeddings], axis=1)
data.x = torch.tensor(combined_features, dtype=torch.float)
print(f"Graph Updated! Features per packet: {data.x.shape[1]}")

Graph Updated! Features per packet: 77


In [25]:
import copy
import torch.optim as optim

# 3. TRAIN GAT_v2 ON COMBINED FEATURES
gat_approach2 = GAT_v2(num_features=data.x.shape[1], num_classes=len(target_enc.classes_))
optimizer2 = optim.Adam(gat_approach2.parameters(), lr=0.009640661422823864, weight_decay=2.5317495991481588e-05)
criterion2 = torch.nn.CrossEntropyLoss()

best_val_acc = 0
best_model_state = None
patience_counter = 0

print("Training GAT_v2 (Approach 2)...")
for epoch in range(1, 2001):
    gat_approach2.train()
    optimizer2.zero_grad()
    out = gat_approach2(data.x, data.edge_index)
    loss = criterion2(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer2.step()

    gat_approach2.eval()
    with torch.no_grad():
        out = gat_approach2(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        val_acc = (pred[data.val_mask] == data.y[data.val_mask]).sum().item() / data.val_mask.sum().item()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(gat_approach2.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 50 == 0:
        print(f'Epoch: {epoch:03d} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}')

    if patience_counter >= 200:
        print(f'\nEarly stopping at epoch {epoch}!')
        break

gat_approach2.load_state_dict(best_model_state)

Training GAT_v2 (Approach 2)...
Epoch: 050 | Loss: 0.2002 | Val Acc: 0.9383
Epoch: 100 | Loss: 0.1843 | Val Acc: 0.9420
Epoch: 150 | Loss: 0.1775 | Val Acc: 0.9432
Epoch: 200 | Loss: 0.1725 | Val Acc: 0.9436
Epoch: 250 | Loss: 0.1641 | Val Acc: 0.9457
Epoch: 300 | Loss: 0.1578 | Val Acc: 0.9449
Epoch: 350 | Loss: 0.1535 | Val Acc: 0.9420
Epoch: 400 | Loss: 0.1513 | Val Acc: 0.9432
Epoch: 450 | Loss: 0.1479 | Val Acc: 0.9428
Epoch: 500 | Loss: 0.1432 | Val Acc: 0.9444

Early stopping at epoch 508!


<All keys matched successfully>

In [26]:
import torch.nn.functional as F
from sklearn.metrics import classification_report

# 1. FINAL RESULTS
gat_approach2.eval()
with torch.no_grad():
    out = gat_approach2(data.x, data.edge_index)
    test_logits = out[data.test_mask]
    test_preds = test_logits.argmax(dim=1).numpy()
    test_true = data.y[data.test_mask].numpy()

final_acc = (test_preds == test_true).sum() / len(test_true)
print(f"\n🏆 GAT_v2 (APPROACH 2) FINAL TEST ACCURACY: {final_acc * 100:.2f}% 🏆\n")

target_names = [str(c) for c in target_enc.classes_]
print("Classification Report:")
print(classification_report(test_true, test_preds, target_names=target_names))


target_local_idx = np.where(idx_test == global_idx)[0][0]

# Extract the raw numbers and the true label for this specific packet
sample_logits = test_logits[target_local_idx]
true_label_id = test_true[target_local_idx]

# Apply softmax to see GAT's exact confidence percentages
sample_probs = F.softmax(sample_logits, dim=0).numpy()
gat_pred = np.argmax(sample_probs)

print("\n" + "="*50)
print(f" APPROACH 2 (GAT_v2) BREAKDOWN FOR TEST SAMPLE {global_idx}")
print("="*50)
print(f"GAT Probabilities     : {np.round(sample_probs, 4)}")
print("-" * 50)
print(f"Final Prediction      : {target_enc.classes_[gat_pred].upper()}")
print(f"Truth                 : {target_enc.classes_[true_label_id].upper()}")
print("="*50)


🏆 GAT_v2 (APPROACH 2) FINAL TEST ACCURACY: 94.69% 🏆

Classification Report:
                 precision    recall  f1-score   support

Data Alteration       0.99      1.00      1.00       139
       Spoofing       0.82      0.30      0.44       168
         normal       0.95      0.99      0.97      2141

       accuracy                           0.95      2448
      macro avg       0.92      0.76      0.80      2448
   weighted avg       0.94      0.95      0.94      2448


 APPROACH 2 (GAT_v2) BREAKDOWN FOR TEST SAMPLE 1012
GAT Probabilities     : [0.     0.0245 0.9755]
--------------------------------------------------
Final Prediction      : NORMAL
Truth                 : NORMAL
